In [4]:
import pandas as pd
from PIL import Image
import torch
from transformers import FlavaProcessor, FlavaModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from tqdm import tqdm
import numpy as np

# Load your CSV
csv_path = "Complete_Img.csv"
df = pd.read_csv(csv_path)

# Drop rows with missing labels, images, or text
# Ensure 'image' and 'text' columns are present
df = df.dropna(subset=['label', 'image', 'text'])

# Map labels to integers
# Ensure 'label' column is present and map labels to integers
df['label'] = df['label'].map({'Non_hope_speech': 0, 'Hope_speech': 1})
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

# Split: 70% train, 15% val, 15% test, stratified
# Ensure stratified split based on labels
#stratify = temp_df['label'] ensures that the split maintains the same proportion of classes in each set.
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=50, shuffle=True,stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=50, shuffle=True,stratify=temp_df['label'])
print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

# Load FLAVA processor and model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = FlavaProcessor.from_pretrained("facebook/flava-full")
model = FlavaModel.from_pretrained("facebook/flava-full").to(device)
model.eval()
#this get_embedding function processes each row to extract the image and text embeddings
# and returns the CLS token embedding normalized.
def get_embedding(row):
    try:
        image = Image.open(row['image']).convert("RGB")
        inputs = processor(image, row['text'], return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            outputs = model(**inputs) # outputs is a tuple
            # Extract the last hidden states
            hidden_states = outputs[0]  # first element is last hidden states
            cls_emb = hidden_states[:, 0, :]  # CLS token embedding
            cls_emb = cls_emb / cls_emb.norm(p=2, dim=-1, keepdim=True)  # normalize
            return cls_emb.cpu().numpy().squeeze(0)
    except Exception as e:# Handle exceptions for image loading or processing
        # Print the error and return None for this row
        print(f"Error processing row {row.name}: {e}")
        return None
#get_embeddings_labels function iterates through the DataFrame,
# extracts embeddings for each row, and collects them along with their labels.
def get_embeddings_labels(df):
    embeddings = []
    labels = []
    for _, row in tqdm(df.iterrows(), total=len(df)):# tqdm provides a progress bar for the loop
        emb = get_embedding(row)
        if emb is not None:# Check if embedding is valid
            embeddings.append(emb)
            labels.append(row['label'])
    return np.vstack(embeddings), np.array(labels)

# Extract embeddings
# and labels for train, validation, and test sets
# train_embs, train_labels = get_embeddings_labels(train_df)
# val_embs, val_labels = get_embeddings_labels(val_df)
# test_embs, test_labels = get_embeddings_labels(test_df)

# # Train logistic regression classifier
# # Using a logistic regression model with class weights to handle class imbalance.
# clf = LogisticRegression(max_iter=1000, random_state=50)
# clf.fit(train_embs, train_labels)

# # Evaluate function
# # This function evaluates the classifier on a given set of embeddings and labels,
# def evaluate(clf, embs, labels, set_name="Set"):
#     preds = clf.predict(embs)# Predict labels using the classifier
#     # Print classification report and metrics
#     print(f"\n{set_name} classification report:")
#     print(classification_report(labels, preds, target_names=["Non", "Yes"], digits=4))
#     print(f"{set_name} Precision: {precision_score(labels, preds):.4f}")
#     print(f"{set_name} Recall:    {recall_score(labels, preds):.4f}")
#     print(f"{set_name} F1 Score:  {f1_score(labels, preds):.4f}")

# # Evaluate on validation and test
# evaluate(clf, val_embs, val_labels, "Validation")
# evaluate(clf, test_embs, test_labels, "Test")


##------------------------Train and Evaulation (RF)--------------------------------##
#Random Forest model for classification
# X_train, y_train = get_embeddings_labels(train_df)
# X_val, y_val = get_embeddings_labels(val_df)
# X_test, y_test = get_embeddings_labels(test_df)

# clf = RandomForestClassifier(n_estimators=250).fit(X_train, y_train)

# y_pred = clf.predict(X_test)

# f1 = f1_score(y_test, y_pred, average='weighted')
# precision = precision_score(y_test, y_pred, average='weighted')
# recall = recall_score(y_test, y_pred, average='weighted')

# print(f"Precision: {precision:.4f}")
# print(f"Recall:    {recall:.4f}")
# print(f"F1 Score:  {f1:.4f}")
# print(classification_report(y_test, y_pred, digits=4))

##------------------------Train and Evaulation (XGB)--------------------------------##
#XGBoost model for classification
#need to translate hope and non hope into 1,0 
X_train, y_train = get_embeddings_labels(train_df)
X_val, y_val = get_embeddings_labels(val_df)
X_test, y_test = get_embeddings_labels(test_df)



clf = XGBClassifier(n_estimators=250).fit(X_train, y_train)

y_predE = clf.predict(X_test)

f1 = f1_score(y_test, y_predE, average='weighted')
precision = precision_score(y_test, y_predE, average='weighted')
recall = recall_score(y_test, y_predE, average='weighted')

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(classification_report(y_test, y_predE, digits=4))

Train size: 2716, Val size: 582, Test size: 583


  0%|          | 0/2716 [00:00<?, ?it/s]C:\Users\lares\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\transformers\modeling_utils.py:1735: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
  1%|          | 21/2716 [00:00<01:26, 31.18it/s]C:\Users\lares\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\PIL\Image.py:3570: UserWarning: image file could not be identified because AVIF support not installed
  warnings.warn(message)
  1%|          | 25/2716 [00:00<01:20, 33.32it/s]

Error processing row 314: cannot identify image file 'images/IHS_0525.jpg'


  2%|▏         | 57/2716 [00:01<01:24, 31.50it/s]

Error processing row 2969: cannot identify image file 'images/IHS_0042.jpg'


...

Error processing row 771: cannot identify image file 'images/IHS_0367.jpg'


 78%|███████▊  | 457/583 [00:16<00:04, 30.96it/s]

Error processing row 2684: cannot identify image file 'images/IHS_0009.jpg'


100%|██████████| 583/583 [00:20<00:00, 27.84it/s]


Precision: 0.9192
Recall:    0.9190
F1 Score:  0.9190
              precision    recall  f1-score   support

           0     0.9094    0.9313    0.9202       291
           1     0.9291    0.9066    0.9177       289

    accuracy                         0.9190       580
   macro avg     0.9192    0.9189    0.9189       580
weighted avg     0.9192    0.9190    0.9190       580

